# UK Police Crime Data Analysis
 loading, cleaning, exploring, and visualizing UK Police Crime Data.
- Import required libraries
- Load the crime dataset
- Clean and explore the data
- Visualize crime locations on a map

In [2]:
# Import Required Libra
import pandas as pd
import plotly.express as px
import os
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

Matplotlib is building the font cache; this may take a moment.


All libraries imported successfully!


## Load the Crime Dataset

 UK Police Crime CSV file in the `data/crime/` directory. 

In [3]:

csv_path = 'data/crime/2025-11-avon-and-somerset-street.csv'
# Load CSV Data
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} rows.")
    display(df.head())
else:
    print(f"File not found: {csv_path}")

Loaded 16977 rows from data/crime/2025-11-avon-and-somerset-street.csv
Columns: ['Crime ID', 'Month', 'Reported by', 'Falls within', 'Longitude', 'Latitude', 'Location', 'LSOA code', 'LSOA name', 'Crime type', 'Last outcome category', 'Context']
Date range: 2025-11 to 2025-11


,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,NaN,2025-11,Avon and Somerset Constabulary,Avon and Somerset Constabulary,-2.492876,51.422716,On or near Trajectus Way,E01014399,Bath and North East Somerset 001A,Anti-social behaviour,NaN,NaN
1,NaN,2025-11,Avon and Somerset Constabulary,Avon and Somerset Constabulary,-2.492876,51.422716,On or near Trajectus Way,E01014399,Bath and North East Somerset 001A,Anti-social behaviour,NaN,NaN
2,55480fb949f84b1a64db63a230f4fb7b4591f04c0251c8...,2025-11,Avon and Somerset Constabulary,Avon and Somerset Constabulary,-2.513901,51.418814,On or near Stockwood Hill,E01014399,Bath and North East Somerset 001A,Burglary,Investigation complete; no suspect identified,NaN
3,eafac600ef57c5d440ee242ec6e89aa3d0823074fa71f1...,2025-11,Avon and Somerset Constabulary,Avon and Somerset Constabulary,-2.509173,51.419104,On or near Durley Hill,E01014399,Bath and North East Somerset 001A,Burglary,Under investigation,NaN
4,a4c40dbd015f83c1023b27dae460cffd68833a8a4561b3...,2025-11,Avon and Somerset Constabulary,Avon and Somerset Constabulary,-2.519070,51.416822,On or near Stockwood Vale,E01014399,Bath and North East Somerset 001A,Criminal damage and arson,Under investigation,NaN



 Dataset Info:
Shape: (16977, 12)
Missing values:
Crime ID                  2054
Month                        0
Reported by                  0
Falls within                 0
Longitude                  741
Latitude                   741
Location                     0
LSOA code                  741
LSOA name                  741
Crime type                   0
Last outcome category     2054
Context                  16977
dtype: int64


## Clean and Explore the Data

Let's check for missing values and basic statistics.

In [5]:
# Check for missing values and show basic info
if 'df' in locals():
    df.info()
    print("\nMissing values per column:")
    df.isnull().sum()
    print("\nCrime types:")
    df['Crime type'].value_counts()

🧹 Original data shape: (16977, 12)
   After dropping rows with missing coordinates: (16236, 12)
   Removed columns: ['Crime ID', 'LSOA code', 'LSOA name', 'Context', 'Last outcome category']
   Created 15 location clusters
✅ Cleaned data shape: (16236, 15)
📈 Features created: ['Month', 'Reported by', 'Falls within', 'Longitude', 'Latitude', 'Location', 'Crime type', 'month_num', 'year', 'location_type', 'crime_severity', 'lat_bin', 'lon_bin', 'location_cluster', 'crime_count']

🎯 Crime Type Distribution:
  Violence and sexual offences: 6404 (39.4%)
  Anti-social behaviour: 2054 (12.7%)
  Public order: 1620 (10.0%)
  Shoplifting: 1525 (9.4%)
  Criminal damage and arson: 1167 (7.2%)
  Other theft: 1035 (6.4%)
  Burglary: 696 (4.3%)
  Vehicle crime: 647 (4.0%)
  Drugs: 272 (1.7%)
  Other crime: 262 (1.6%)

⚠️ Crime Severity Distribution:
  high: 6764 (41.7%)
  low: 6690 (41.2%)
  medium: 2782 (17.1%)


,Month,Reported by,Falls within,Longitude,Latitude,Location,Crime type,month_num,year,location_type,crime_severity,lat_bin,lon_bin,location_cluster,crime_count
0,2025-11,Avon and Somerset Constabulary,Avon and Somerset Constabulary,-2.492876,51.422716,On or near Trajectus Way,Anti-social behaviour,11,2025,street,low,51.42,-2.49,12,4
1,2025-11,Avon and Somerset Constabulary,Avon and Somerset Constabulary,-2.492876,51.422716,On or near Trajectus Way,Anti-social behaviour,11,2025,street,low,51.42,-2.49,12,4
2,2025-11,Avon and Somerset Constabulary,Avon and Somerset Constabulary,-2.513901,51.418814,On or near Stockwood Hill,Burglary,11,2025,street,medium,51.42,-2.51,12,10
3,2025-11,Avon and Somerset Constabulary,Avon and Somerset Constabulary,-2.509173,51.419104,On or near Durley Hill,Burglary,11,2025,street,medium,51.42,-2.51,12,10
4,2025-11,Avon and Somerset Constabulary,Avon and Somerset Constabulary,-2.519070,51.416822,On or near Stockwood Vale,Criminal damage and arson,11,2025,street,medium,51.42,-2.52,8,1


## Visualize Crime Locations on a Map

We'll plot the crime locations using Plotly. Only rows with valid latitude and longitude will be shown.

In [7]:
# Visualize Crime Locations on a Map
print("Visualizing Crime Locations on Map...")

# Check if dataframe exists
if 'df' in locals() or 'df' in globals():
    # Make a clean copy to avoid SettingWithCopyWarning
    df_map = df.dropna(subset=['Latitude', 'Longitude']).copy()
    
    print(f"📊 Plotting {len(df_map)} locations with valid coordinates")
    print(f"📍 Coordinate range: Lat [{df_map['Latitude'].min():.3f}, {df_map['Latitude'].max():.3f}], "
          f"Lon [{df_map['Longitude'].min():.3f}, {df_map['Longitude'].max():.3f}]")
    
    # Count crimes by type for the legend
    crime_counts = df_map['Crime type'].value_counts()
    print("\n📈 Crimes by type (top 10):")
    for crime_type, count in crime_counts.head(10).items():
        print(f"  {crime_type}: {count}")
    
    # Create the map visualization
    try:
        # Try using the newer scatter_map (Plotly 5.16+)
        fig = px.scatter_map(
            df_map,
            lat="Latitude",
            lon="Longitude",
            color="Crime type",
            hover_data=["Crime type", "Month", "Location", "Reported by"],
            hover_name="Crime type",
            zoom=9,
            height=700,
            title=f"Crime Locations in Avon and Somerset ({len(df_map)} incidents)",
            size_max=10
        )
        # Use map_style for scatter_map
        fig.update_layout(map_style="open-street-map")
    except:
        # Fall back to scatter_mapbox for older Plotly versions
        print("⚠️ Using scatter_mapbox (scatter_map not available)")
        fig = px.scatter_mapbox(
            df_map,
            lat="Latitude",
            lon="Longitude",
            color="Crime type",
            hover_data=["Crime type", "Month", "Location", "Reported by"],
            hover_name="Crime type",
            zoom=9,
            height=700,
            title=f"Crime Locations in Avon and Somerset ({len(df_map)} incidents)"
        )
        fig.update_layout(mapbox_style="open-street-map")
    
    # Update layout for better visualization
    fig.update_layout(
        margin={"r":0,"t":50,"l":0,"b":0},
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=0.01,
            bgcolor='rgba(255, 255, 255, 0.8)',
            font=dict(size=10)
        ),
        hoverlabel=dict(
            bgcolor="white",
            font_size=12,
            font_family="Arial"
        )
    )
    
    # Add a marker for map center
    center_lat = df_map['Latitude'].mean()
    center_lon = df_map['Longitude'].mean()
    
    fig.update_layout(
        map=dict(
            center=dict(lat=center_lat, lon=center_lon)
        )
    )
    
    # Show the figure
    print("\n✅ Map created! Displaying interactive visualization...")
    fig.show()
    
    # Additional statistics
    print("\n" + "="*60)
    print("📍 LOCATION STATISTICS")
    print("="*60)
    
    # Most common locations
    print("\n🏢 Top 10 Most Common Crime Locations:")
    top_locations = df_map['Location'].value_counts().head(10)
    for location, count in top_locations.items():
        print(f"  {location}: {count} incidents")
    
    # Geographic clustering
    print(f"\n🌍 Geographic Coverage:")
    print(f"  Northernmost: {df_map['Latitude'].max():.4f}°N")
    print(f"  Southernmost: {df_map['Latitude'].min():.4f}°N")
    print(f"  Easternmost: {df_map['Longitude'].max():.4f}°E")
    print(f"  Westernmost: {df_map['Longitude'].min():.4f}°E")
    
    # Save the figure as HTML for sharing
    fig.write_html("crime_locations_map.html")
    print(f"\n💾 Map saved as 'crime_locations_map.html' (open in browser to interact)")
    
    # Create a simplified version by crime severity
    print("\n" + "="*60)
    print("⚠️ CRIME SEVERITY VISUALIZATION")
    print("="*60)
    
    # Define crime severity
    def classify_severity(crime_type):
        violent_crimes = ['violence', 'assault', 'robbery', 'sexual', 'weapon']
        property_crimes = ['burglary', 'theft', 'vehicle', 'shoplifting', 'damage']
        
        crime_lower = str(crime_type).lower()
        if any(v in crime_lower for v in violent_crimes):
            return 'High'
        elif any(p in crime_lower for p in property_crimes):
            return 'Medium'
        else:
            return 'Low'
    
    df_map['Severity'] = df_map['Crime type'].apply(classify_severity)
    
    # Create severity map
    severity_colors = {'High': 'red', 'Medium': 'orange', 'Low': 'green'}
    
    fig_severity = px.scatter_map(
        df_map,
        lat="Latitude",
        lon="Longitude",
        color="Severity",
        color_discrete_map=severity_colors,
        hover_data=["Crime type", "Month", "Location", "Severity"],
        zoom=9,
        height=600,
        title="Crime Locations by Severity Level"
    )
    
    fig_severity.update_layout(map_style="open-street-map")
    fig_severity.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
    
    print("\n🔴 High Severity: Violence, Assault, Robbery, Sexual offenses, Weapons")
    print("🟠 Medium Severity: Burglary, Theft, Vehicle crime, Criminal damage")
    print("🟢 Low Severity: Anti-social behaviour, Public order, Other crimes")
    
    print("\n📊 Severity Distribution:")
    severity_counts = df_map['Severity'].value_counts()
    for severity, count in severity_counts.items():
        percentage = (count / len(df_map)) * 100
        print(f"  {severity}: {count} incidents ({percentage:.1f}%)")
    
    fig_severity.show()
    fig_severity.write_html("crime_severity_map.html")
    print("💾 Severity map saved as 'crime_severity_map.html'")
    
else:
    print("❌ No DataFrame found. Please run the data loading cell first.")
   

🔧 Preparing data for ML model...
   Target classes: ['high', 'low', 'medium']
   Encoded: Reported by
   Encoded: Falls within
   Encoded: location_type
✅ Features shape: (16236, 9)
✅ Target shape: (16236,)

🎯 Target distribution:
  high: 6764 samples (41.7%)
  low: 6690 samples (41.2%)
  medium: 2782 samples (17.1%)

📋 Features used: ['Latitude', 'Longitude', 'month_num', 'year', 'crime_count', 'Reported by_encoded', 'Falls within_encoded', 'location_type_encoded', 'location_cluster']


## Machine Learning: Predicting Crime Type
We'll use the cleaned data to train a model that predicts the type of crime based on available features. We'll select the best features, encode categorical variables, and use a robust classifier (Random Forest) for prediction.

In [10]:
# Cell 5: Train ML Model (No matplotlib version)
def train_crime_model_no_plot(X, y, test_size=0.2, random_state=42):
    """Train Random Forest model for crime prediction without matplotlib"""
    
    print("🤖 Training ML model...")
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    
    print(f"   Training set: {X_train.shape}")
    print(f"   Test set: {X_test.shape}")
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train Random Forest
    rf_model = RandomForestClassifier(
        n_estimators=150,
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=random_state,
        class_weight='balanced',
        n_jobs=-1
    )
    
    print("   Training Random Forest...")
    rf_model.fit(X_train_scaled, y_train)
    
    # Predictions
    y_pred = rf_model.predict(X_test_scaled)
    y_pred_proba = rf_model.predict_proba(X_test_scaled)
    
    # Evaluation
    print("\n" + "="*60)
    print("📊 MODEL EVALUATION")
    print("="*60)
    
    accuracy = accuracy_score(y_test, y_pred)
    print(f"✅ Accuracy: {accuracy:.3f}")
    
    print("\n📋 Classification Report:")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
    
    print("\n🎯 Confusion Matrix:")
    conf_matrix = confusion_matrix(y_test, y_pred)
    print(conf_matrix)
    
    # Feature importance
    print("\n" + "="*60)
    print("🔝 FEATURE IMPORTANCE")
    print("="*60)
    
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("Top 15 Most Important Features:")
    for i, row in importance_df.head(15).iterrows():
        print(f"  {row['feature']}: {row['importance']:.4f}")
    
    # Create text-based visualization
    print("\n📊 Feature Importance (text visualization):")
    print("-" * 50)
    for i, row in importance_df.head(10).iterrows():
        bar_length = int(row['importance'] * 100)
        print(f"{row['feature'][:30]:30} | {'█' * bar_length} ({row['importance']:.3f})")
    
    return rf_model, scaler, X_test_scaled, y_test, y_pred, X_train, X_test

# Train the model
if 'X' in locals() and 'y' in locals():
    model, scaler, X_test_scaled, y_test, y_pred, X_train, X_test = train_crime_model_no_plot(X, y)
    print("\n✅ Model training completed!")

🤖 Training ML model...
   Training set: (12988, 9)
   Test set: (3248, 9)
   Training Random Forest...

📊 MODEL EVALUATION
✅ Accuracy: 0.488

📋 Classification Report:
              precision    recall  f1-score   support

        high       0.55      0.53      0.54      1353
         low       0.58      0.51      0.54      1338
      medium       0.25      0.35      0.29       557

    accuracy                           0.49      3248
   macro avg       0.46      0.46      0.46      3248
weighted avg       0.51      0.49      0.50      3248


🎯 Confusion Matrix:
[[717 325 311]
 [385 676 277]
 [203 161 193]]

🔝 FEATURE IMPORTANCE
Top 15 Most Important Features:
  Latitude: 0.4005
  Longitude: 0.3892
  crime_count: 0.1745
  location_cluster: 0.0357
  month_num: 0.0000
  year: 0.0000
  Reported by_encoded: 0.0000
  Falls within_encoded: 0.0000
  location_type_encoded: 0.0000

📊 Feature Importance (text visualization):
--------------------------------------------------
Latitude            

AttributeError: module 'matplotlib' has no attribute 'backends'

## Predict Crime Type for New Data
You can now use the trained model to predict the type of crime for new/unseen data. Below is an example of how to make predictions using the trained model.